# Screening against manual labels of search results

LLM screening of 400 records drawn from the literature search, scored against the authors' title-and-abstract labels
(`data/consistency-check-datasets/screening/search-sample/search-sample.csv`).
Predictions are derived with the screening pipeline in `evals_local`. ELIGIBLE is the positive class.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path("../..").resolve())) 
from evals_local.loaders import load_screening_predictions
from evals_local.run import run_task
from evals_local.normalizers import normalize_screening_label

## 1. Derive pred labels from the batch outputs (no truth yet)

`load_screening_predictions` merges the batch records with the LLM outputs on `UT`, reads the
per-stage results `s1_r..s4_r`, and sets `pred = NOT_ELIGIBLE` if any stage is 0 else `ELIGIBLE`.

In [2]:
RUN = "l0_cc1_search-sample_010726_medium_f1"
TRUTH_PATH = Path("../../data/consistency-check-datasets/screening/search-sample/search-sample.csv")

REPO_ROOT = Path("../..").resolve()
eval_summary = run_task("screening", RUN, label_path=TRUTH_PATH)
print(f"saved eval data: {Path(eval_summary['data_path']).relative_to(REPO_ROOT)}")
for label, path in eval_summary["metric_paths"].items():
    print(f"saved eval metrics ({label}): {Path(path).relative_to(REPO_ROOT)}")


preds = load_screening_predictions(RUN)
preds["custom_id"] = preds["UT"].astype(str).str.strip()
preds["pred_label"] = preds["pred"].map(normalize_screening_label)

print(f"prediction rows: {len(preds)}")
assert preds["custom_id"].is_unique, "duplicate UT in predictions"
preds["pred_label"].value_counts()

saved eval data: data/labels/eval/l0_cc1_search-sample_010726_medium_f1/data/screening.xlsx
saved eval metrics (eligibility): data/labels/eval/l0_cc1_search-sample_010726_medium_f1/metrics/screening_eligibility_label_metrics.xlsx
saved eval metrics (eligibility_confusion): data/labels/eval/l0_cc1_search-sample_010726_medium_f1/metrics/screening_eligibility_confusion.xlsx
prediction rows: 400


pred_label
NOT_ELIGIBLE    231
ELIGIBLE        169
Name: count, dtype: int64

## 2. Load human truth and join on WOS ID (grain guarded)

The truth `eligibility` column is normalized with the *same* normalizer as the prediction so both
sides use the identical `ELIGIBLE` / `NOT_ELIGIBLE` format. The join asserts every prediction finds
exactly one truth row (no missing matches, no grain mismatch).

In [3]:
truth = pd.read_csv(TRUTH_PATH)
truth["custom_id"] = truth["custom_id"].astype(str).str.strip()
truth["true_label"] = truth["eligibility"].map(normalize_screening_label)

assert truth["custom_id"].is_unique, "duplicate custom_id in truth (grain mismatch)"

df = preds.merge(
    truth[["custom_id", "true_label"]],
    on="custom_id",
    how="inner",
    validate="one_to_one",
)

unmatched = len(preds) - len(df)
assert unmatched == 0, f"{unmatched} predictions had no matching truth row"
print(f"joined rows: {len(df)} | empty truth: {int(df['true_label'].eq('').sum())}")
df[["pred_label", "true_label"]].value_counts().reset_index(name="count")

joined rows: 400 | empty truth: 0


,pred_label,true_label,count
0,NOT_ELIGIBLE,NOT_ELIGIBLE,219
1,ELIGIBLE,ELIGIBLE,109
2,ELIGIBLE,NOT_ELIGIBLE,60
3,NOT_ELIGIBLE,ELIGIBLE,12


## 3. Confusion matrix + precision / recall / F1 (ELIGIBLE = positive)

In [4]:
scored = df[df["true_label"] != ""].copy()

pred_pos = scored["pred_label"] == "ELIGIBLE"
true_pos = scored["true_label"] == "ELIGIBLE"

tp = int((pred_pos & true_pos).sum())
fp = int((pred_pos & ~true_pos).sum())
fn = int((~pred_pos & true_pos).sum())
tn = int((~pred_pos & ~true_pos).sum())

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

confusion = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index=pd.Index(["pred ELIGIBLE", "pred NOT_ELIGIBLE"], name="prediction"),
    columns=pd.Index(["true ELIGIBLE", "true NOT_ELIGIBLE"], name="truth"),
)
confusion

truth,true ELIGIBLE,true NOT_ELIGIBLE
prediction,,
pred ELIGIBLE,109,60
pred NOT_ELIGIBLE,12,219


In [5]:
pd.DataFrame([
    {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_support": tp + fn,
        "scored_rows": len(scored),
    }
])

,tp,fp,fn,tn,precision,recall,f1,positive_support,scored_rows
0,109,60,12,219,0.64497,0.900826,0.751724,121,400
